Separate file into
1. SARS-CoV-2 Mpro
2. MERS-CoV Mpro
3. some mysterious ones that is undefined maybe? 

In [69]:
import os
import subprocess
import re

def alignment(TMalign_path, ref_pdb, query_path, seq_ID = 1):
    
    results = []

    # regex patterns for TM-align output
    seqid_pattern = re.compile(r"\Seq_ID=.*?=\s*([\d\.]+)")
    tmscore_pattern = re.compile(r"TM-score.*?=\s*([\d\.]+)")

    for query_id in os.listdir(query_path):
        query_dir = os.path.join(query_path, query_id)
        if not os.path.isdir(query_dir):
            continue
    
        query_pdb = os.path.join(query_dir, f"{query_id}.pdb")
        if not os.path.isfile(query_pdb):
            continue
    
        # run TM-align
        proc = subprocess.run([TMalign_path, ref_pdb, query_pdb], capture_output=True, text=True)

        if proc.returncode != 0:
            continue

        stdout = proc.stdout

        # parse Seq_ID and TM-score
        seqid_match = seqid_pattern.search(stdout)
        tmscore_match = tmscore_pattern.search(stdout)

        if not seqid_match or not tmscore_match:
            continue

        seq_id = float(seqid_match.group(1))
        tm_score = float(tmscore_match.group(1))

        # store only non-identical structures
        if seq_id >= seq_ID:
            results.append({
                "pdb_id": query_id,
                "Seq_ID": seq_id,
                "TM_score": tm_score
            })

    return results


In [13]:
TMalign = "/Users/hannah/tools/TMalign/TMalign"
SARS_ref = "../aligned_files/7gef-a/7gef-a.pdb" #highest resolution SARS_CoV_2 Mpro
aligned_files = '../aligned_files'

In [61]:
SARS_CoV_2 = alignment(TMalign, SARS_ref, aligned_files, seq_ID=0.9)

In [62]:
print(len(SARS_CoV_2))

1320


In [63]:
check = []
for complex in SARS_CoV_2:
    if complex['Seq_ID'] != 1:
        check.append(complex["pdb_id"])
    else:
        continue
print(check)

['SARS-a1133a', 'SARS-a1133b', '7gjx-a']


In [ ]:
#append all SARS-Mpro in one folder

import shutil

def movefolder(src_root, dst_root, ID_dic): 
    '''ID_dic from alignment in dictionary formate, including pdb_id, TM_score and Seq_ID'''
    os.makedirs(dst_root, exist_ok=True)

    count = 0

    for complex in ID_dic:
        complex_id = complex['pdb_id']
        complex_dir = os.path.join(src_root, complex_id)
        if not os.path.isdir(complex_dir):
            continue

        pdb_path = os.path.join(complex_dir, f"{complex_id}.pdb")
        if not os.path.isfile(pdb_path):
            continue

        dst_path = os.path.join(dst_root, f"{complex_id}.pdb")
        shutil.copy2(pdb_path, dst_path)
        count += 1

    print(f"Copied {count} PDB files to {dst_root}")

In [68]:
root = "../aligned_files"
sars_dst_root = "../SARS_Mpro"

movefolder(root, sars_dst_root, SARS_CoV_2)

Copied 1320 PDB files to ../SARS_Mpro


In [74]:
#now for the MERS-data
MERS_ref = "../aligned_files/MERS-a0080a/MERS-a0080a.pdb"

MERS_CoV = alignment(TMalign, MERS_ref, aligned_files, seq_ID=0.9)

In [75]:
print(len(MERS_CoV))

594


In [ ]:
check_MERS = []
for complex in MERS_CoV:
    if complex['Seq_ID'] != 1:
        check_MERS.append(complex["pdb_id"])
        check_MERS.append(complex['Seq_ID'])
    else:
        continue
print(check_MERS)

#all the Seq_ID are really high, even for the ones that didn't show up as 1, potentially just intrinsic movements
#gonna blast a few just to double check

['MERS-a1795b', 0.997, 'MERS-a2106a', 0.997, 'MERS-a1549b', 0.997, 'MERS-a1696b', 0.997, 'MERS-a1015b', 0.993, 'MERS-a1795a', 0.997, 'MERS-a2106b', 0.997, 'MERS-a1549a', 0.997, 'MERS-a1696a', 0.997, 'MERS-a1015a', 0.993]
